# 🚀 HVAC-RL 快速测试 (5-10分钟)

这个笔记本用于快速验证环境是否正常工作。

## ✅ 测试内容
1. GPU检查
2. 环境设置
3. BEAR仿真器测试
4. 简短的PPO训练（1000步）

**总耗时**: 约5-10分钟

## Step 1: 检查GPU

In [ ]:
import torch

print("=" * 60)
print("GPU信息")
print("=" * 60)
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU显存: {gpu_mem:.1f} GB")
    print("✅ GPU已就绪！")
else:
    print("❌ 未检测到GPU")
    print("请设置: Runtime > Change runtime type > GPU")

## Step 2: 克隆项目并安装依赖

In [ ]:
import os

# 克隆项目
if not os.path.exists("/content/HVAC-RL"):
    print("克隆项目...")
    !git clone https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git /content/HVAC-RL
else:
    print("项目已存在，拉取最新更新...")
    !cd /content/HVAC-RL && git pull

os.chdir("/content/HVAC-RL")
print(f"✅ 当前目录: {os.getcwd()}")

In [ ]:
# 安装依赖（这可能需要2-3分钟）
print("安装依赖包...")
!pip install -q torch transformers accelerate peft
!pip install -q stable-baselines3 gymnasium
!pip install -q numpy pandas scikit-learn matplotlib seaborn
!pip install -q pvlib cvxpy tqdm

print("\n✅ 依赖安装完成！")

## Step 3: 验证环境

In [ ]:
# 运行环境验证脚本
!python verify_environment.py

## Step 4: 测试BEAR环境

In [ ]:
# 运行BEAR环境测试
!python test_bear_env.py

## Step 5: 快速PPO训练测试（1000步，约2-3分钟）

In [ ]:
# 创建测试输出目录
import os
os.makedirs("/content/test_output", exist_ok=True)

# 运行快速PPO训练（只训练1000步用于测试）
print("开始快速PPO训练测试（1000步）...")
print("预计耗时: 2-3分钟\n")

# 设置环境变量
import os
os.environ["BUILDING"] = "OfficeLarge"
os.environ["WEATHER"] = "Hot_Dry"
os.environ["LOCATION"] = "Tucson"
os.environ["TOTAL_STEPS"] = "1000"
os.environ["SAVE_DIR"] = "/content/test_output"

# 运行PPO训练
!cd /content/HVAC-RL && python core_modules/ppo_collect.py

## Step 6: 检查测试结果

In [ ]:
import os
from pathlib import Path

print("=" * 60)
print("测试结果")
print("=" * 60)

test_dir = Path("/content/test_output")
expected_files = [
    "ppo_trajectory.json",
    "ppo_final.zip",
    "training_metrics.json",
    "ppo_training_results.png"
]

all_good = True
for fname in expected_files:
    fpath = test_dir / fname
    if fpath.exists():
        size = fpath.stat().st_size / 1024
        print(f"✅ {fname} ({size:.1f} KB)")
    else:
        print(f"❌ {fname} (未找到)")
        all_good = False

print("=" * 60)
if all_good:
    print("\n🎉 所有测试通过！环境配置正确！")
    print("\n你现在可以运行完整的Pipeline了：")
    print("  选项1: 使用 Colab_A100_Complete_Pipeline.ipynb")
    print("  选项2: 直接运行命令")
    print("    python core_modules/main_pipeline.py --stage all")
    print("\n📊 完整Pipeline将自动生成以下可视化报告：")
    print("  - Stage 1: PPO训练分析（奖励曲线、统计分布等）")
    print("  - Stage 2: Few-shot样本选择分析")
    print("  - Stage 3: LLM推理分析（解析成功率、奖励对比）")
    print("  - Stage 4: 自蒸馏数据分析（过滤前后对比）")
    print("  - Stage 5: 微调进度可视化")
    print("  - Stage 6: 最终方法对比（PPO vs LLM vs 微调后LLM）")
    print("  所有报告保存在: pipeline_output/reports/")
else:
    print("\n⚠️  部分文件缺失，请检查上面的错误信息")

In [ ]:
# 显示训练结果图表
from PIL import Image
import matplotlib.pyplot as plt

plot_path = "/content/test_output/ppo_training_results.png"
if os.path.exists(plot_path):
    img = Image.open(plot_path)
    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PPO Training Results (1000 steps test)', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("训练结果图表未找到")

---
## ✅ 测试完成！

如果所有测试都通过了，你可以：

### 下一步：运行完整Pipeline

打开完整版笔记本：
```
Colab_A100_Complete_Pipeline.ipynb
```

或者直接在这里运行完整Pipeline：

In [ ]:
# 运行完整Pipeline（可选，需要4-8小时）
# 取消下面的注释来运行

# print("⚠️  这将运行完整Pipeline，预计需要4-8小时")
# print("如果你确定要运行，取消下面命令的注释\n")

# !python core_modules/main_pipeline.py --stage all --building OfficeLarge --weather Hot_Dry

## 🔧 故障排除

### 如果GPU检查失败
1. Runtime > Change runtime type
2. Hardware accelerator 选择 "GPU"
3. GPU type 选择 "A100" (如果可用)

### 如果BEAR环境测试失败
1. 检查数据文件是否存在：`!ls BEAR/Data/`
2. 重新克隆项目
3. 重新安装依赖

### 如果PPO训练失败
1. 检查错误信息
2. 验证环境变量是否正确设置
3. 确保有足够的磁盘空间：`!df -h`